# OPRB: chạy một model trên Kaggle hoặc Colab

1. Sửa `MODEL`, `STAGE` và `SAMPLE` ở ô cấu hình.
2. Kaggle: bật **Internet**; chọn **GPU T4 x2** cho docTR và hai VLM, **không GPU** cho `tesseract` và `ppocrv5_mobile_en`. Riêng `latency`: chọn **GPU T4 x2 cho cả 5 model** để đo trên cùng loại máy (hai model CPU không dùng GPU).
3. Chạy dài: **Save Version → Save & Run All**. Kết quả nằm ở tab Output của phiên bản đó.
4. Chấm lại (không cần GPU, không nạp model): `STAGE = "score"`, **không GPU**, Add Input → Notebook Output của phiên `run`.
5. Chạy tiếp phiên bị ngắt: **Add Input → Notebook Output** của phiên trước, rồi chạy lại với cùng cấu hình.

Thứ tự: `pilot` → `run` với `SAMPLE = 200` để soát ảnh lỗi, rỗng, bị cắt → chốt `configs/models.toml` → `run` với `SAMPLE = 3000` → `latency`.

Mẫu nhỏ là phần đầu của mẫu lớn, nên ảnh đã xong không phải chạy lại, miễn là không đổi `configs/models.toml`.
Model nào không kịp 3.000 ảnh thì chấm cả 5 model ở `SAMPLE = 2000`.

In [ ]:
# ===== CẤU HÌNH =====
MODEL = "smoke"     # smoke | tesseract | ppocrv5_mobile_en | doctr_fast_parseq | lightonocr2_1b | qwen3_vl_2b
STAGE = "pilot"     # pilot | run | latency | score
COMMIT = "main"     # khi chạy thật: dùng tag hoặc mã commit đã chốt, không dùng main
BATCH_SIZE = None   # None = mặc định theo model
SAMPLE = 3000       # tổng số ảnh test cho STAGE = "run", lấy ngẫu nhiên theo tỉ lệ gốc (2 text : 1 nontext)
HOURS = 11.0        # dừng sớm để Kaggle kịp lưu kết quả (phiên tối đa 12 giờ)

In [ ]:
import os, pathlib, shutil, subprocess, sys, time

START = time.time()
KAGGLE = pathlib.Path("/kaggle").exists()
COLAB = "google.colab" in sys.modules
if KAGGLE:
    FAST, KEEP = pathlib.Path("/kaggle/temp"), pathlib.Path("/kaggle/working/results")
elif COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    FAST, KEEP = pathlib.Path("/content"), pathlib.Path("/content/drive/MyDrive/oprb_results")
else:
    raise SystemExit("Notebook này chạy trên Kaggle hoặc Colab.")
REPO, DATA = FAST / "oprb-eval", FAST / "oprb_test"
FAST.mkdir(parents=True, exist_ok=True)
KEEP.mkdir(parents=True, exist_ok=True)
os.environ.update(OPRB_CACHE=str(FAST / "cache"), HF_HOME=str(FAST / "hf"), PYTHONUNBUFFERED="1")

def sh(command, **kwargs):
    print("$", command, flush=True)
    subprocess.run(command, shell=True, check=True, **kwargs)

if not REPO.exists():
    sh(f"git clone -q https://github.com/conthonamanco/oprb-eval.git {REPO}")
sh(f"git fetch -q --tags origin && git checkout -q {COMMIT}", cwd=REPO)
if COMMIT == "main":
    sh("git merge -q --ff-only origin/main", cwd=REPO)
sh("git log -1 --oneline", cwd=REPO)
GPUS = subprocess.run("nvidia-smi -L", shell=True, capture_output=True, text=True).stdout.strip().splitlines()
print("GPU:", GPUS or "không có", "| CPU:", os.cpu_count())

In [ ]:
EXTRA = {"smoke": "", "tesseract": "tesseract", "ppocrv5_mobile_en": "ppocr",
         "doctr_fast_parseq": "doctr", "lightonocr2_1b": "vlm", "qwen3_vl_2b": "vlm"}[MODEL]
if STAGE == "score":
    EXTRA = ""  # scoring reads saved predictions and GT only
CPU_ONLY = MODEL in ("smoke", "tesseract", "ppocrv5_mobile_en")
if STAGE != "score" and (not CPU_ONLY or (STAGE == "latency" and MODEL != "smoke")) and not GPUS:
    raise SystemExit("Cần GPU T4 x2 (model này, hoặc stage latency): đổi Accelerator rồi chạy lại.")

def torch_version():
    r = subprocess.run([sys.executable, "-c", "import torch; print(torch.__version__)"], capture_output=True, text=True)
    return r.stdout.strip()

before = torch_version()
extras = ",".join(x for x in ("data", "verify", EXTRA) if x)
sh(f'{sys.executable} -m pip install -q -e "{REPO}[{extras}]"')
after = torch_version()
print("torch:", before, "->", after)
if before and after != before:
    raise SystemExit("pip đã thay torch của máy. Dừng lại và ghim phiên bản thư viện.")
sh(f"{sys.executable} -m unittest discover -s tests", cwd=REPO)

In [ ]:
# Khôi phục kết quả của phiên trước (Kaggle: Add Input -> Notebook Output)
if KAGGLE:
    for old in sorted(pathlib.Path("/kaggle/input").glob(f"**/results/{MODEL}/*")):
        target = KEEP / MODEL / old.name
        target.parent.mkdir(parents=True, exist_ok=True)
        # Nhiều phiên cũ có thể chứa cùng một file: file chỉ nối thêm nên bản lớn nhất là bản mới nhất
        if old.is_file() and (not target.exists() or old.stat().st_size > target.stat().st_size):
            shutil.copy(old, target)
            print("khôi phục", old)

PY = f"{sys.executable} -m oprb_eval"
CONFIG = "--config configs/smoke.toml" if MODEL == "smoke" else ""
PATHS = f"--data {DATA} --results {KEEP}"
sh(f"{PY} download --data {DATA}", cwd=REPO)
if STAGE != "pilot":
    sh(f"{PY} stats {PATHS}", cwd=REPO)
    sh(f"{PY} stats {PATHS} --sample {SAMPLE}", cwd=REPO)

In [ ]:
# Nạp model một lần trước khi chia shard, để các tiến trình không cùng tải trọng số
# Pilot chỉ kiểm tra kỹ thuật (lỗi, tốc độ, dtype) trên vài ảnh test và không đọc GT
if STAGE == "pilot":
    options = {
        "lightonocr2_1b": "--images 8 --variants dtype=float32 dtype=float16 --batch-sizes 4 8",
        "qwen3_vl_2b": "--images 8 --variants dtype=float32 dtype=float16 dtype=bfloat16 --batch-sizes 4",
    }.get(MODEL, "--images 24")
else:
    options = "--images 1"
warm = PATHS if STAGE == "pilot" else f"--data {DATA} --results {FAST / 'warm'}"
if STAGE != "score":
    sh(f"{PY} pilot --model {MODEL} {CONFIG} {warm} {options}", cwd=REPO)

In [ ]:
DEFAULT_BATCH = {"lightonocr2_1b": 8, "qwen3_vl_2b": 4, "doctr_fast_parseq": 4}
batch = BATCH_SIZE or DEFAULT_BATCH.get(MODEL, 1)

if STAGE == "run":
    shards = (os.cpu_count() or 1) if CPU_ONLY else max(1, len(GPUS))
    sample = f"--sample {SAMPLE}"
    processes = []
    for i in range(shards):
        env = dict(os.environ)
        if CPU_ONLY:
            env.update(CUDA_VISIBLE_DEVICES="", OMP_THREAD_LIMIT="1", OMP_NUM_THREADS="1", OPRB_THREADS="1")
        else:
            env["CUDA_VISIBLE_DEVICES"] = str(i)
        command = f"{PY} run --model {MODEL} {CONFIG} {PATHS} {sample} --shard {i}/{shards} --batch-size {batch}"
        print("$", command, flush=True)
        processes.append(subprocess.Popen(command, shell=True, cwd=REPO, env=env, start_new_session=True))

    stopped = False
    while any(p.poll() is None for p in processes):
        time.sleep(20)
        if time.time() - START > HOURS * 3600:
            import signal
            for p in processes:
                if p.poll() is None:
                    os.killpg(p.pid, signal.SIGTERM)
            stopped = True
            print("Hết giờ: đã dừng và giữ kết quả. Chạy lại notebook này để tiếp tục.")
            break
    for p in processes:
        p.wait()
    if not stopped:
        if any(p.returncode for p in processes):
            raise SystemExit(f"Có shard bị lỗi: {[p.returncode for p in processes]}")
        sh(f"{PY} score --model {MODEL} {PATHS} {sample}", cwd=REPO)

if STAGE == "score":
    sh(f"{PY} score --model {MODEL} {PATHS} --sample {SAMPLE}", cwd=REPO)

if STAGE == "latency":
    # Một tiến trình, một GPU, từng ảnh một
    env = dict(os.environ, CUDA_VISIBLE_DEVICES="" if CPU_ONLY else "0")
    sh(f"{PY} latency --model {MODEL} {CONFIG} {PATHS} --sample 100", cwd=REPO, env=env)

In [ ]:
sh(f"ls -la {KEEP} {KEEP / MODEL} 2>/dev/null || true")
print(f"Tổng thời gian: {(time.time() - START) / 3600:.2f} giờ")